In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Load cleaned data
df = pd.read_csv("../data/cleaned/ipo_cleaned.csv")
df["Date"] = pd.to_datetime(df["Date"])

# Fix the remaining column name
df = df.rename(columns={"Current  Gains (%)": "current_gains_pct"})

print(f"Shape: {df.shape}")
print("Data loaded successfully!")
df.head(3)

Shape: (264, 20)
Data loaded successfully!


,Date,ipo_name,issue_size_cr,qib_subscription,hni_subscription,rii_subscription,total_subscription,issue_price,listing_open,listing_close,listing_gains_pct,cmp,current_gains_pct,Year,Month,Quarter,listing_gain_abs,performance_category,nifty_close,nifty_30d_return
0,2010-02-03,Infinite Comp,189.8,48.44,106.02,11.08,43.22,165.0,178.35,184.51,11.82,473.75,187.12,2010,2,1,19.51,Moderate,4931.85,NaN
1,2010-02-08,Jubilant Food,328.7,59.39,51.95,3.79,31.11,145.0,160.00,114.50,-21.03,3770.20,2500.14,2010,2,1,-30.50,Loss,4760.40,NaN
2,2010-02-15,Vascon Engineer,199.8,1.12,3.65,0.62,1.22,165.0,155.90,146.38,-11.28,21.70,-86.85,2010,2,1,-18.62,Loss,4801.95,NaN


In [2]:
yearly = df.groupby("Year").size().reset_index(name="ipo_count")

fig = px.bar(
    yearly,
    x="Year",
    y="ipo_count",
    title="Number of IPOs per Year (2010–2021)",
    color="ipo_count",
    color_continuous_scale="teal",
    text="ipo_count"
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

In [3]:
fig = px.histogram(
    df,
    x="listing_gains_pct",
    nbins=40,
    title="Distribution of IPO Listing Gains (%)",
    color_discrete_sequence=["#1D9E75"],
    marginal="box"
)
fig.add_vline(x=0, line_dash="dash", line_color="red",
              annotation_text="Break-even")
fig.add_vline(x=df["listing_gains_pct"].mean(),
              line_dash="dot", line_color="orange",
              annotation_text=f"Mean: {df['listing_gains_pct'].mean():.1f}%")
fig.show()

In [4]:
cat_counts = df["performance_category"].value_counts().reset_index()
cat_counts.columns = ["category", "count"]

color_map = {
    "Blockbuster": "#1D9E75",
    "Strong"     : "#5DCAA5",
    "Moderate"   : "#FAC775",
    "Weak"       : "#F0997B",
    "Loss"       : "#E24B4A"
}

fig = px.pie(
    cat_counts,
    names="category",
    values="count",
    title="IPO Performance Category Breakdown",
    color="category",
    color_discrete_map=color_map,
    hole=0.4
)
fig.show()

In [5]:
fig = px.scatter(
    df,
    x="total_subscription",
    y="listing_gains_pct",
    color="performance_category",
    size="issue_size_cr",
    hover_name="ipo_name",
    title="Total Subscription vs Listing Gains",
    labels={
        "total_subscription": "Subscription (x times)",
        "listing_gains_pct" : "Listing Gain (%)"
    },
    color_discrete_map=color_map
)
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

In [6]:
yearly_gains = df.groupby("Year")["listing_gains_pct"].agg(
    ["mean", "median", "std"]
).reset_index()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=yearly_gains["Year"],
    y=yearly_gains["mean"],
    name="Mean Gain %",
    marker_color="#1D9E75"
))

fig.add_trace(go.Scatter(
    x=yearly_gains["Year"],
    y=yearly_gains["median"],
    name="Median Gain %",
    mode="lines+markers",
    line=dict(color="#FAC775", width=2)
))

fig.update_layout(
    title="Year-wise Average IPO Listing Gains",
    xaxis_title="Year",
    yaxis_title="Listing Gain (%)",
    barmode="group"
)
fig.show()

In [10]:
fig = px.scatter(
    df.dropna(subset=["nifty_30d_return"]),
    x="nifty_30d_return",
    y="listing_gains_pct",
    color="performance_category",
    hover_name="ipo_name",
    title="Market Mood (Nifty 30d Return) vs IPO Listing Gains",
    labels={
        "nifty_30d_return"  : "Nifty 30-Day Return (%) at IPO Date",
        "listing_gains_pct" : "IPO Listing Gain (%)"
    },
    trendline="ols",
    color_discrete_map=color_map
)
fig.show()

In [11]:
numeric_cols = [
    "issue_size_cr", "qib_subscription", "hni_subscription",
    "rii_subscription", "total_subscription", "issue_price",
    "listing_gains_pct", "nifty_30d_return"
]

corr = df[numeric_cols].corr()

fig = px.imshow(
    corr,
    title="Correlation Heatmap — IPO Features",
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    text_auto=".2f"
)
fig.show()

In [12]:
top10 = df.nlargest(10, "listing_gains_pct")[["ipo_name", "listing_gains_pct"]]
bot10 = df.nsmallest(10, "listing_gains_pct")[["ipo_name", "listing_gains_pct"]]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Top 10 IPOs by Listing Gain",
                                    "Bottom 10 IPOs by Listing Gain"])

fig.add_trace(go.Bar(
    x=top10["listing_gains_pct"],
    y=top10["ipo_name"],
    orientation="h",
    marker_color="#1D9E75",
    name="Top 10"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=bot10["listing_gains_pct"],
    y=bot10["ipo_name"],
    orientation="h",
    marker_color="#E24B4A",
    name="Bottom 10"
), row=1, col=2)

fig.update_layout(height=500, title_text="Best and Worst IPOs (2010–2021)")
fig.show()

In [13]:
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}

monthly = df.groupby("Month")["listing_gains_pct"].mean().reset_index()
monthly["Month_Name"] = monthly["Month"].map(month_names)

fig = px.bar(
    monthly,
    x="Month_Name",
    y="listing_gains_pct",
    title="Average IPO Listing Gains by Month (Seasonality)",
    color="listing_gains_pct",
    color_continuous_scale="RdYlGn",
    text=monthly["listing_gains_pct"].round(1)
)
fig.update_traces(textposition="outside")
fig.show()